# 05 — SHAP Explainability

Why does the model predict "aggressive" or "drowsy"?

This notebook uses **SHAP (SHapley Additive exPlanations)** to explain the predictions of the best classical ML model (Random Forest) trained in notebook 03.

- Global feature importance (beeswarm plot)
- Per-class SHAP bar summary
- Single-window explanation (waterfall plot)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path

from models import load_model
from features import get_feature_columns
from sklearn.preprocessing import StandardScaler

PROC_DIR   = Path('../data/processed')
MODELS_DIR = Path('../results/models')
FIG_DIR    = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
LABEL_NAMES = ['normal', 'aggressive', 'drowsy']

## 1. Load Data & Best Model

In [ ]:
df        = pd.read_parquet(PROC_DIR / 'features_5s_2s.parquet')
feat_cols = get_feature_columns(df)

# Last fold: train on D1-D5, test on D6
fold_idx    = 5
test_driver = 'D6'

train_idx = df[df['driver'] != test_driver].index
test_idx  = df[df['driver'] == test_driver].index

X_train = df.loc[train_idx, feat_cols].values.astype(float)
X_test  = df.loc[test_idx,  feat_cols].values.astype(float)
y_test  = df.loc[test_idx,  'label'].values.astype(int)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model_path = MODELS_DIR / f'RandomForest_fold{fold_idx}_test{test_driver}.joblib'
model = load_model(model_path)
print(f'Model loaded: {model_path.name}')
print(f'Test set size: {len(X_test_sc)} windows (driver {test_driver})')

## 2. SHAP Values (TreeExplainer)

In [ ]:
# TreeExplainer is fast and exact for tree-based models
explainer = shap.TreeExplainer(model)

# Balanced subsample for speed (max 500 windows)
rng        = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test_sc), size=min(500, len(X_test_sc)), replace=False)
X_sample   = X_test_sc[sample_idx]
feat_df    = pd.DataFrame(X_sample, columns=feat_cols)

shap_values = explainer.shap_values(feat_df)  # list: one array per class
print(f'SHAP values shape per class: {shap_values[0].shape}')
print(f'Classes: {LABEL_NAMES}')

## 3. Global Feature Importance — Beeswarm Plot

Each dot is one window. Colour = feature value (red=high, blue=low).  
X-axis = SHAP value (positive = pushes toward this class).

In [ ]:
for cls_idx, cls_name in enumerate(LABEL_NAMES):
    shap.summary_plot(
        shap_values[cls_idx], feat_df,
        max_display=20,
        show=False,
        plot_type='dot',
    )
    plt.title(f'SHAP Beeswarm — {cls_name.capitalize()} driving', fontsize=13)
    plt.tight_layout()
    plt.gcf().savefig(FIG_DIR / f'shap_beeswarm_{cls_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Bar Summary — Mean |SHAP| per Class

In [ ]:
colors = {'normal': '#4C72B0', 'aggressive': '#DD8452', 'drowsy': '#55A868'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for cls_idx, (cls_name, ax) in enumerate(zip(LABEL_NAMES, axes)):
    mean_abs = np.abs(shap_values[cls_idx]).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:15]
    ax.barh(
        [feat_cols[i] for i in top_idx][::-1],
        mean_abs[top_idx][::-1],
        color=colors[cls_name]
    )
    ax.set_title(f'{cls_name.capitalize()} — Top 15 Features', fontsize=11)
    ax.set_xlabel('Mean |SHAP value|')

plt.suptitle('Feature Impact by Driving Class', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / 'shap_bar_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Single-Window Explanation (Waterfall)

Explains why the model classified one specific window as **aggressive**.

In [ ]:
# Find an aggressive window in our sample
y_sample = y_test[sample_idx]
aggressive_mask = (y_sample == 1)

if aggressive_mask.sum() == 0:
    print('No aggressive windows in sample — using index 0')
    window_idx = 0
else:
    window_idx = np.where(aggressive_mask)[0][0]

pred_class = model.predict(X_sample[[window_idx]])[0]
print(f'True label : {LABEL_NAMES[y_sample[window_idx]]}')
print(f'Predicted  : {LABEL_NAMES[pred_class]}')

# Waterfall plot for the predicted class
exp = shap.Explanation(
    values      = shap_values[pred_class][window_idx],
    base_values = explainer.expected_value[pred_class],
    data        = X_sample[window_idx],
    feature_names = feat_cols,
)
shap.plots.waterfall(exp, max_display=15, show=False)
plt.title(f'Waterfall — predicted: {LABEL_NAMES[pred_class]}', fontsize=11)
plt.tight_layout()
plt.gcf().savefig(FIG_DIR / 'shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Insights

| Finding | Interpretation |
|---|---|
| **Jerk features dominate aggressive class** | High rate of acceleration change is the strongest signal of aggressive driving |
| **Lateral G-force** | Sharp lane changes create lateral acceleration spikes |
| **Speed variance** | Frequent speed fluctuation correlates with aggressive behaviour |
| **Drowsy class hardest** | Drowsy driving is smooth and slow — heavily overlaps with normal at the signal level |